In [ ]:
#review

import pandas as pd

def process_reviews(input_file='reviews.csv', output_file='review.csv'):
    # Đọc dữ liệu thô
    df = pd.read_csv(input_file)

    df_clean = pd.DataFrame()

    # 1. Review_ID
    if 'review_id' in df.columns:
        df_clean['Review_ID'] = df['review_id'].astype(str).str.strip()
    elif 'Review_ID' in df.columns:
        df_clean['Review_ID'] = df['Review_ID'].astype(str).str.strip()
    else:
        df_clean['Review_ID'] = ['REV-' + str(i + 1).zfill(7) for i in range(len(df))]

    # 2. Rating (giới hạn từ 1 đến 5 sao)
    rating_col = 'rating' if 'rating' in df.columns else 'Rating'
    if rating_col in df.columns:
        df_clean['Rating'] = pd.to_numeric(df[rating_col], errors='coerce').fillna(5).astype(int).clip(1, 5)
    else:
        df_clean['Rating'] = 5

    # 3. Review_Text (tự động ánh xạ từ review_text hoặc review_title)
    if 'review_text' in df.columns:
        df_clean['Review_Text'] = df['review_text'].fillna('').astype(str).str.strip()
    elif 'review_title' in df.columns:
        df_clean['Review_Text'] = df['review_title'].fillna('').astype(str).str.strip()
    else:
        df_clean['Review_Text'] = ''

    # 4. Review_Date
    date_col = 'review_date' if 'review_date' in df.columns else 'Review_Date'
    if date_col in df.columns:
        df_clean['Review_Date'] = pd.to_datetime(df[date_col], errors='coerce').dt.strftime('%Y-%m-%d %H:%M:%S')
    else:
        df_clean['Review_Date'] = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')

    # 5. Customer_ID
    cust_col = 'customer_id' if 'customer_id' in df.columns else 'Customer_ID'
    df_clean['Customer_ID'] = df[cust_col].astype(str).str.strip() if cust_col in df.columns else 'UNKNOWN'

    # 6. Product_ID
    prod_col = 'product_id' if 'product_id' in df.columns else 'Product_ID'
    df_clean['Product_ID'] = df[prod_col].astype(str).str.strip() if prod_col in df.columns else 'UNKNOWN'

    # Lọc trùng lặp và sắp xếp đúng 6 cột theo lược đồ REVIEW
    cols = ['Review_ID', 'Rating', 'Review_Text', 'Review_Date', 'Customer_ID', 'Product_ID']
    review_table = df_clean[cols].drop_duplicates(subset=['Review_ID']).dropna(subset=['Review_ID'])

    # Xuất file CSV
    review_table.to_csv(output_file, index=False, encoding='utf-8')
    print(f"-> Xuất thành công bảng REVIEW: {output_file} ({len(review_table):,} dòng)")

if __name__ == '__main__':
    process_reviews()

In [ ]:
import pandas as pd

def process_product_ratings(input_file='reviews.csv', output_file='product_ratings.csv'):
    df = pd.read_csv(input_file)

    prod_col = 'product_id' if 'product_id' in df.columns else 'Product_ID'
    rating_col = 'rating' if 'rating' in df.columns else 'Rating'
    review_col = 'review_id' if 'review_id' in df.columns else 'Review_ID'

    df[rating_col] = pd.to_numeric(df[rating_col], errors='coerce').fillna(5).clip(1, 5)

    # Gom nhóm theo Product_ID
    ratings_summary = df.groupby(prod_col).agg(
        Average_Rating=(rating_col, 'mean'),
        Total_Reviews=(review_col, 'count')
    ).reset_index()

    ratings_summary.rename(columns={prod_col: 'Product_ID'}, inplace=True)
    ratings_summary['Average_Rating'] = ratings_summary['Average_Rating'].round(2)

    ratings_summary.to_csv(output_file, index=False, encoding='utf-8')
    print(f"-> Xuất thành công bảng PRODUCT_RATINGS: {output_file} ({len(ratings_summary):,} dòng)")

if __name__ == '__main__':
    process_product_ratings()

In [ ]:
import pandas as pd

def process_customer_reviews(input_file='reviews.csv', output_file='customer_reviews.csv'):
    df = pd.read_csv(input_file)

    cust_col = 'customer_id' if 'customer_id' in df.columns else 'Customer_ID'
    rating_col = 'rating' if 'rating' in df.columns else 'Rating'
    review_col = 'review_id' if 'review_id' in df.columns else 'Review_ID'

    df[rating_col] = pd.to_numeric(df[rating_col], errors='coerce').fillna(5).clip(1, 5)

    # Gom nhóm theo Customer_ID
    cust_summary = df.groupby(cust_col).agg(
        Review_Count=(review_col, 'count'),
        Avg_Rating_Given=(rating_col, 'mean')
    ).reset_index()

    cust_summary.rename(columns={cust_col: 'Customer_ID'}, inplace=True)
    cust_summary['Avg_Rating_Given'] = cust_summary['Avg_Rating_Given'].round(2)

    cust_summary.to_csv(output_file, index=False, encoding='utf-8')
    print(f"-> Xuất thành công bảng CUSTOMER_REVIEWS: {output_file} ({len(cust_summary):,} dòng)")

if __name__ == '__main__':
    process_customer_reviews()